# CMRI_Model — matched-step A/B pilot comparison
**General → A (General replay + Ugi) vs B (General replay + Ugi with 5% LNPDB in the lipid stream)**. Compare matched 5,000-update checkpoints. No model training or automatic job submission. CPU log inventory is enabled; GPU inference, new sampled evaluation LMDBs and Ridge probes are opt-in. Original training artifacts are read-only.

**Hypothesis:** LNPDB may act as auxiliary chemical diversity augmentation (not corrupted labels or coordinates). Compare domain transfer, Ugi retention and general retention. The two-arm design cannot establish a regularization mechanism; only the effect of replacing Ugi exposure with LNPDB under this run's settings. One training seed cannot establish between-run reproducibility.

**Execution:** Run sections 0–2 first. Confirm checkpoint, mixture, and held-out split provenance. Section 3 prepares matched panels (only new files under OUT). Section 4 runs the original H200 evaluator adapted to use a common single-source evaluation task and checks actual input hashes. Section 5 summarizes paired results. Section 6 optionally reuses the original independent chemistry pipeline. Use heart_gpu on an allocated GPU for inference. All heavy switches default False; no source data/checkpoints modified.

**Location:** Store this notebook under `validation/notebooks/`. All newly generated results are written under `validation/evidence/cmri_model_ab_comparison/`; existing training and general evaluation inputs remain unchanged.


## 0 | Paths and switches
Edit only overrides and flags as needed. Known paths are from the previous project notebooks and pilot discussion; LNPDB validation location is deliberately not guessed. A checkpoint named `last` is not assumed to be at update 5,000: inspect metadata before evaluation.
Execution switches are ON. Ugi/LNPDB held-out provenance gates remain OFF until actual pilot exclusions are verified. Training-settings review is bypassed at user request; logged differences are still descriptive.


In [ ]:
from pathlib import Path
from argparse import Namespace
import os, sys, re, json, hashlib, pickle, random, math
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE=Path('/project/yli4/ypan')
HOME=Path('/home/ypan/heart_liver/model_train')
PILOT=BASE/'heart_pilot_runs/continual'
RUNS={'A':PILOT/'heart-pilot-a-pilot','B':PILOT/'heart-pilot-b-pilot'}
GENERAL=BASE/'runs/h200/full/heart-h200-b448-v1/checkpoint_best.pt'
GENERAL_DATA=BASE/'UniMolReference/ligand_data/ligands'
AUDIT=BASE/'heart_evaluation/general_audit_10k_v2'
GENERAL_MANIFEST=AUDIT/'validation_10k_seed17'
UGI=BASE/'lipid_library/lmdb'
VALIDATION=HOME/'validation'
OUT=VALIDATION/'evidence/cmri_model_ab_comparison'
OUT.mkdir(parents=True,exist_ok=True)
REPO=next((x for x in [HOME/'heart-molecular-pretraining-v2',Path('/scratch/ypan/heart_liver/model_train/heart-molecular-pretraining-v2')] if (x/'model/unimol/tasks/unimol_contrastive.py').is_file()), HOME/'heart-molecular-pretraining-v2')
UNICORE=next((x for x in [HOME/'third_party/Uni-Core',BASE/'third_party/Uni-Core',Path('/scratch/ypan/heart_liver/model_train/third_party/Uni-Core')] if x.is_dir()),HOME/'third_party/Uni-Core')
CHECKPOINT_OVERRIDE={'A':None,'B':None}  # absolute file path if multiple candidates
LOG_OVERRIDE={'A':None,'B':None}         # absolute log path if multiple logs
LNP_VALID_LMDB=None                     # actual training-excluded validation LMDB (single file)
LNP_SPLIT_EVIDENCE=None                  # path to actual split manifest/audit documenting B exclusion, not proposal
LNP_HOLDOUT_REVIEWED=False               # set only after inspecting evidence and confirming canonical train exclusion
UGI_SPLIT_REVIEWED=False                 # set after checking actual pilot train/valid shard routing against splits.json
TRAIN_SETTINGS_REVIEWED=True            # user requests evaluation without repeating manual settings gate; causal attribution still requires audit
RUN_CHECKPOINT_METADATA=True            # CPU checkpoint loads; trusted files only
PREPARE_DOMAIN_PANELS=True              # copies only sampled valid records to OUT
RUN_FROZEN_EVAL=True                    # GPU inference, no training
RUN_DOWNSTREAM=True                     # GPU embeddings + CPU Ridge, no encoder retraining
N_EVAL=1000                              # matched per-source target; use 192 for initial test or 10000 later
BATCH=8; EVAL_SEED=13; SAMPLE_SEED=17
assert N_EVAL>=BATCH and N_EVAL%BATCH==0, 'Use a multiple of BATCH to avoid unequal negative pools'
EVAL_REFERENCE_CHECKPOINT=GENERAL
for name,path in {'General checkpoint':GENERAL,'General manifest':GENERAL_MANIFEST,'Ugi root':UGI,'Repo':REPO,'Uni-Core':UNICORE,**RUNS}.items():print(name,path,path.exists())
print('Outputs:',OUT)
RUN_CHECKPOINT_CURVES=True
# UGI_SPLIT_REVIEWED and LNP_HOLDOUT_REVIEWED remain evidence gates, not execution switches.


## 1 | Inventory, training reports and actual logged metrics (CPU)
The A/B logs may use different mixture distributions. Loss is descriptive only unless evaluated on the same examples with the same loss settings. Inventory never scans molecular source directories. Multiple training logs are displayed, not silently combined.

In [ ]:
def read_json(p):return json.loads(Path(p).read_text(encoding="utf-8"))
def sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
CHECKPOINTS={'General':GENERAL}; LOGS={}; inventory=[]
for m,root in RUNS.items():
    print('\n',m,root)
    if not root.is_dir(): print('MISSING run directory');continue
    for p in sorted(root.rglob('*')):
        if p.is_file() and (p.suffix in {'.pt','.json','.log','.out','.csv'}):
            inventory.append({'model':m,'path':str(p),'bytes':p.stat().st_size});print(p.relative_to(root))
    exact=sorted(root.rglob('checkpoint*_5000.pt'))
    last=sorted(root.rglob('checkpoint_last.pt'))
    cand=exact if exact else last
    ck=Path(CHECKPOINT_OVERRIDE[m]) if CHECKPOINT_OVERRIDE[m] else (cand[0] if len(cand)==1 else None)
    if ck and ck.is_file(): CHECKPOINTS[m]=ck
    else:print('Select CHECKPOINT_OVERRIDE explicitly from inventory:',cand)
    logs=sorted(root.rglob('train*.log'))
    log=Path(LOG_OVERRIDE[m]) if LOG_OVERRIDE[m] else (logs[0] if len(logs)==1 else None)
    if log and log.is_file(): LOGS[m]=[log]
    elif LOG_OVERRIDE[m] is None:
        LOGS[m]=logs or sorted(root.rglob('*.out')) or sorted(root.rglob('*.log'))
    else: print('Invalid LOG_OVERRIDE:',log)
    for name in ['continual_init_report.json','mixture_report.json']:
        for p in sorted(root.rglob(name)):
            print(name,p);display(read_json(p))
pd.DataFrame(inventory).to_csv(OUT/'run_inventory.csv',index=False)
print('Selected checkpoints:',CHECKPOINTS)

In [ ]:
# Support Uni-Core JSON progress records and pipe-delimited/text logs. Preserve raw lines.
def parse_log(path,model):
    rows=[]
    for lineno,line in enumerate(Path(path).open(encoding='utf-8',errors='replace'),1):
        vals={}
        for hit in re.finditer(r'\{[^{}]*\}',line):
            try: vals.update(json.loads(hit.group()))
            except (ValueError,TypeError):pass
        if not vals:
            vals={k:v for k,v in re.findall(r'\b([A-Za-z][\w-]*)\s*[:= ]\s*(-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)',line)}
        phase='valid' if re.search(r'\bvalid(?:_inner)?\b',line,re.I) or any(k.startswith('valid_') for k in vals) else ('train' if re.search(r'\btrain(?:_inner)?\b',line,re.I) or any(k.startswith('train_') for k in vals) else 'unknown')
        norm={}
        for k,v in vals.items():
            k=re.sub(r'^(train|valid)(?:_inner)?_','',k).replace('-','_')
            try: norm[k]=float(v)
            except (ValueError,TypeError):pass
        step=next((norm[k] for k in ['num_updates','updates','update','step'] if k in norm),None)
        if step is None:
            match=re.search(r'\b(?:num_updates|updates|update|step)\s*[:= ]\s*(\d+)',line)
            if match: step=float(match.group(1))
        if 'loss' not in norm and not any('loss' in k for k in norm):continue
        rows.append({'model':model,'phase':phase,'line':lineno,'step':step,'source':str(path),'raw':line.strip(),**{k:v for k,v in norm.items() if k not in ['model','phase','line','step','raw']}})
    return rows
logs=pd.DataFrame([r for m,paths in LOGS.items() for p in paths for r in parse_log(p,m)])
logs.to_csv(OUT/'parsed_pilot_logs.csv',index=False)
if len(logs):
    display(logs.groupby(['model','phase'],dropna=False).tail(2).drop(columns='raw',errors='ignore'))
    print('Rows without explicit update:',logs.step.isna().sum())
else:print('No supported log metrics parsed; inspect original log lines and update parser, do not infer results.')
from matplotlib import font_manager
FONT=next((f for f in ['Arial','Liberation Sans','DejaVu Sans'] if f in {x.name for x in font_manager.fontManager.ttflist}), 'DejaVu Sans')
plt.rcParams.update({'font.family':FONT,'font.size':12,'axes.titlesize':12,'axes.labelsize':12,'xtick.labelsize':12,'ytick.labelsize':12,'legend.fontsize':12,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
COLORS={'General':'#7C546D','A':'#E6A34C','B':'#B54A55'}
def save(fig,name):
    from matplotlib.text import Text
    fig.canvas.draw()
    for t in fig.findobj(match=Text):t.set_fontfamily(FONT);t.set_fontsize(15 if t is getattr(fig,'_suptitle',None) else 12)
    f=OUT/f'{name}.svg';fig.savefig(f,bbox_inches='tight',facecolor='white');f.write_text(f.read_text(encoding='utf-8').replace(FONT,'Arial'),encoding='utf-8');plt.show();plt.close(fig)
if len(logs) and 'loss' in logs:
    fig,axs=plt.subplots(1,2,figsize=(12,4),layout='constrained')
    for ax,phase in zip(axs,['train','valid']):
        for m in ['A','B']:
            q=logs[(logs.model==m)&(logs.phase==phase)].dropna(subset=['step','loss']).sort_values('step')
            if len(q):ax.plot(q.step,q.loss,label=m,color=COLORS[m],marker='.',ms=2)
        ax.set(title=phase.capitalize()+' logged loss (descriptive)',xlabel='Updates',ylabel='Recorded loss')
        if ax.lines:ax.legend(frameon=False)
    if any(ax.lines for ax in axs):save(fig,'CMRI_Model_AB_training_loss')
    else:plt.close(fig)
# Always run log plots, independent of GPU and panel switches.
if not logs.empty:
    log_metrics=[k for k in logs.columns if 'loss' in k and pd.api.types.is_numeric_dtype(logs[k])]
    for metric in log_metrics:
        fig,axs=plt.subplots(1,2,figsize=(12,4),layout='constrained')
        for ax,phase in zip(axs,['train','valid']):
            for m in ['A','B']:
                q=logs[(logs.model==m)&(logs.phase==phase)].dropna(subset=['step',metric]).sort_values('step').drop_duplicates(['step',metric])
                if len(q):ax.plot(q.step,q[metric],label=m,color=COLORS[m],marker='.',ms=3)
            ax.set(title=f'{phase}: {metric}',xlabel='Updates',ylabel=metric)
            if ax.lines:ax.legend(frameon=False)
            else:ax.text(.5,.5,'No logged update-indexed data',ha='center',transform=ax.transAxes)
        save(fig,'logged_'+metric)
    display(logs.groupby(['model','phase']).size().rename('records'))


## 2 | Verify matched checkpoints and settings (CPU, optional)
Load trusted checkpoint metadata only. Original task/loss/architecture from the baseline are reused for all frozen evaluations to avoid silently using the pilot mixed-source loader. Strict state loading must succeed; if the pilot changed architecture, stop and adapt evaluator explicitly. Confirm actual 5000 update values, training configs and mixture reports; no claim of matched settings until reviewed.

In [ ]:
VERIFIED_UPDATES={}
if RUN_CHECKPOINT_METADATA or RUN_FROZEN_EVAL or RUN_DOWNSTREAM:
    import torch
    for m,p in CHECKPOINTS.items():
        if not p.is_file():print('Missing',m,p);continue
        ck=torch.load(p,map_location='cpu',weights_only=False)
        args=ck.get('args',{});args=vars(args) if hasattr(args,'__dict__') else args
        history=ck.get('optimizer_history') or []
        updates=history[-1].get('num_updates') if history and isinstance(history[-1],dict) else None
        if updates is None:updates=ck.get('num_updates')
        VERIFIED_UPDATES[m]=updates
        print(m,'updates:',updates,'keys:',list(ck),'args:')
        display({k:v for k,v in args.items() if any(s in k.lower() for s in ['seed','batch','update','lr','task','loss','arch','replay','mix','data','temperature','restore','reset','mask','noise'])})
        del ck
    print('Verified update counts:',VERIFIED_UPDATES)
else:print('Checkpoint metadata disabled. Enable RUN_CHECKPOINT_METADATA to verify actual updates.')

## 3 | Build fixed held-out evaluation panels (opt-in CPU)
General: reuse the original 10K manifest and take the same first N records for all models. Ugi: sample uniformly across numeric records in declared validation shards (weighted by shard sizes). LNPDB: use only a verified actual held-out LMDB, never the whole library or an unverified proposed split. Copy sampled payloads without changing molecular coordinates into a new single-source LMDB for the original evaluator. Preserve source-key provenance and hash the exact sampled records. This uses original general-task augmentation rather than the mixed loader. Domain panels are descriptive until split provenance is reviewed; General pretraining overlap and near-duplicate/scaffold leakage are not excluded by this step.

In [ ]:
from shutil import copy2
PANEL_ROOT=OUT/'evaluation_panels'/f'n{N_EVAL}_seed{SAMPLE_SEED}'
PANELS={'General':(GENERAL_DATA,GENERAL_MANIFEST,N_EVAL)}
def extract_smi(rec):
    for k in ['smi','smiles','SMILES','canonical_smiles']:
        v=rec.get(k,rec.get(k.encode()))
        if isinstance(v,bytes):v=v.decode()
        if isinstance(v,str) and v.strip():return v.strip()
    raise ValueError('Missing molecule SMILES')
def make_panel(name,files,counts,proof):
    import lmdb
    from rdkit import Chem
    d=PANEL_ROOT/name; manifest=d/'sample_metadata.csv'; info=d/'sampling_summary.json'
    if info.is_file():
        prior=read_json(info)
        if prior['source_proof']!=proof or prior['n']!=N_EVAL or sha(manifest)!=prior['manifest_sha256'] or sha(d/'valid.lmdb')!=prior['panel_sha256'] or sha(d/'dict.txt')!=prior['dict_sha256']:raise RuntimeError('Existing panel provenance changed; inspect before reuse')
        return d,d,N_EVAL
    if d.exists():raise RuntimeError(f'Partial panel exists: {d}. Inspect before removing or rebuilding.')
    if sum(counts)<N_EVAL:raise ValueError(f'{name}: fewer than {N_EVAL} valid records; lower N_EVAL for all domains')
    idx=np.random.default_rng(SAMPLE_SEED).choice(sum(counts),N_EVAL,replace=False);ends=np.cumsum(counts); starts=np.r_[0,ends[:-1]]
    grouped=defaultdict(list)
    for j,i in enumerate(idx):
        f=int(np.searchsorted(ends,i,side='right'));grouped[f].append((j,int(i-starts[f])))
    d.mkdir(parents=True); copy2(GENERAL_DATA/'dict.txt',d/'dict.txt')
    records=[]; out=lmdb.open(str(d/'valid.lmdb'),subdir=False,map_size=16*1024**3)
    try:
        for f,items in grouped.items():
            src=lmdb.open(str(files[f]),subdir=False,readonly=True,lock=False,readahead=False)
            try:
                with src.begin() as tx,out.begin(write=True) as dst:
                    if tx.stat()['entries']!=counts[f]:raise ValueError('Shard count mismatch')
                    for j,k in items:
                        payload=tx.get(str(k).encode());
                        if payload is None:raise KeyError((files[f],k))
                        rec=pickle.loads(payload);sm=extract_smi(rec); mol=Chem.MolFromSmiles(sm)
                        if mol is None:raise ValueError('Invalid SMILES')
                        dst.put(str(j).encode(),payload)
                        records.append({'sample_order':j,'lmdb_index':j,'lmdb_key':str(j),'smiles':sm,'canonical_smiles':Chem.MolToSmiles(mol,isomericSmiles=True),'source_lmdb':str(files[f]),'source_key':k,'payload_sha256':hashlib.sha256(payload).hexdigest()})
            finally:src.close()
    finally:out.close()
    frame=pd.DataFrame(records).sort_values('sample_order');frame.to_csv(manifest,index=False)
    print(name,'sampled records:',len(frame),'canonical duplicates:',frame.canonical_smiles.duplicated().sum())
    info.write_text(json.dumps({'n':N_EVAL,'source_lmdb_records':N_EVAL,'source_proof':proof,'manifest_sha256':sha(manifest),'panel_sha256':sha(d/'valid.lmdb'),'dict_sha256':sha(d/'dict.txt')},indent=2))
    return d,d,N_EVAL
if PREPARE_DOMAIN_PANELS:
    import lmdb
    if UGI_SPLIT_REVIEWED:
        m=read_json(UGI/'manifest.json');s=read_json(UGI/'splits.json'); valid=sorted(s['valid']);train=set(s['train'])
        assert valid and not set(valid)&train
        sizes={x['lmdb']:int(x['n_records']) for x in m['shards']};assert set(valid)<=sizes.keys()
        PANELS['Ugi']=make_panel('Ugi',[UGI/x for x in valid],[sizes[x] for x in valid],{'manifest':sha(UGI/'manifest.json'),'splits':sha(UGI/'splits.json')})
    else:print('Ugi skipped: review actual pilot split routing first.')
    if LNP_HOLDOUT_REVIEWED and LNP_VALID_LMDB and LNP_SPLIT_EVIDENCE:
        f=Path(LNP_VALID_LMDB);proof=Path(LNP_SPLIT_EVIDENCE);assert f.is_file() and proof.is_file()
        env=lmdb.open(str(f),subdir=False,readonly=True,lock=False,readahead=False)
        try:n=env.stat()['entries']
        finally:env.close()
        PANELS['LNPDB']=make_panel('LNPDB',[f],[n],{'source':str(f.resolve()),'source_sha256':sha(f),'split_evidence':sha(proof)})
    else:print('LNPDB skipped: verified actual B-held-out LMDB and split evidence required.')
else:print('Domain panel preparation disabled.')
# Reuse previously completed panels only after revalidating provenance via the section above.
print(PANELS)

## 4 | Frozen evaluator (original H200 implementation, adapted)
The next cell defines the original evaluator without invoking it. Changes: common baseline task configuration, explicit RNG seeds and fingerprint of actual collated inputs (including targets). Same batch size, sample order, augmentation, and criterion are mandatory. Do not compare objective results if input hashes differ. Original normalized projector assumption is separately checked below. Source: HEART_H200_One_Notebook_Project_Results, cell 17.

In [ ]:
"""Read-only HEART checkpoint diagnostics and paired-view molecular embeddings.

Scientific objectives (same HEART model + original task.build_loss):
- masked atom: exact token accuracy over MASKED positions, not overall atoms;
- geometry: coord/distance original losses, plus denoising vs noisy-input baseline;
- NT-Xent: original criterion loss, positive/negative cosine, in-batch top1;
- representations: raw encoder CLS and projected normalized CLS (both views).

Uses HEART numeric-key manifest; checks source SMILES vs task sample names.
Only trusted checkpoints: torch.load(...,weights_only=False) unpickles the checkpoint.
"""
import argparse
import csv
from collections import Counter
from datetime import datetime, timezone
import hashlib
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
import numpy as np

def heart_eval_now():
    return datetime.now(timezone.utc).isoformat()

def heart_eval_sha256_file(path: Path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(4 << 20), b''):
            h.update(block)
    return h.hexdigest()

def heart_eval_move(x, device):
    import torch
    if isinstance(x, torch.Tensor):
        return x.to(device, non_blocking=True)
    if isinstance(x, dict):
        return {k: heart_eval_move(v, device) for k, v in x.items()}
    if isinstance(x, tuple):
        return tuple((heart_eval_move(v, device) for v in x))
    if isinstance(x, list):
        return [heart_eval_move(v, device) for v in x]
    return x

def heart_eval_float_scalar(x):
    import torch
    if isinstance(x, torch.Tensor):
        return float(x.detach().item()) if x.numel() == 1 else None
    if isinstance(x, (float, int, np.number)):
        return float(x)
    return None

def heart_eval_git_info(repo):
    result = {}
    try:
        for name, args in {'head': ['git', '-C', str(repo), 'rev-parse', 'HEAD'], 'status': ['git', '-C', str(repo), 'status', '--porcelain']}.items():
            result[name] = subprocess.run(args, text=True, capture_output=True, timeout=10, check=True).stdout.strip()
    except Exception as exc:
        result['error'] = str(exc)
    return result

def heart_eval_avg(x):
    return float(np.mean(x)) if len(x) else None

def heart_eval_main(args_in):
    ap = argparse.ArgumentParser(description=__doc__)
    for name in ['repo', 'data-dir', 'checkpoint', 'manifest-dir', 'out']:
        ap.add_argument('--' + name, required=True, type=Path)
    ap.add_argument('--unicore-dir', type=Path)
    ap.add_argument('--limit', type=int, default=0, help='0 = entire fixed manifest; 8 = smoke test; 192 = paired baseline comparison')
    ap.add_argument('--batch-size', type=int, default=16)
    ap.add_argument('--seed', type=int, default=13, help='Random seed for model baseline (not data sampling)')
    ap.add_argument('--variant', choices=['trained', 'random_init'], default='trained')
    ap.add_argument('--cpu', action='store_true')
    ap.add_argument('--skip-embeddings', action='store_true', help='Keep objective diagnostics only')
    args = args_in
    if args.batch_size < 2 or args.limit < 0:
        ap.error('Require batch-size>=2 for contrastive negatives and limit>=0')
    expected = args.repo / 'model' / 'unimol'
    for path in (expected, args.data_dir / f'{args.split}.lmdb', args.data_dir / 'dict.txt', args.manifest_dir / 'sample_metadata.csv', args.manifest_dir / 'sampling_summary.json'):
        if not path.exists():
            ap.error(f'Missing {path}')
    if args.variant == 'trained' and (not args.checkpoint.is_file()):
        ap.error(f'Checkpoint missing: {args.checkpoint}')
    args.out.mkdir(parents=True, exist_ok=True)
    for p in ['objective_batches.csv', 'molecule_diagnostics.csv', 'evaluation_summary.json']:
        if (args.out / p).exists():
            ap.error(f'Output {args.out / p} already exists. Use separate run folder to avoid overwritten evidence.')
    if args.unicore_dir and args.unicore_dir.is_dir():
        sys.path.insert(0, str(args.unicore_dir.resolve()))
    sys.path.insert(0, str((args.repo / 'model').resolve()))
    import pandas as pd
    import torch
    from unicore import tasks
    import unicore
    import unimol
    print('unimol:', unimol.__file__, 'unicore:', unicore.__file__, flush=True)
    if expected.resolve() not in Path(unimol.__file__).resolve().parents:
        raise RuntimeError('Wrong UniMol module imported: ' + unimol.__file__)
    device = torch.device('cpu' if args.cpu else 'cuda:0')
    if device.type == 'cuda' and (not torch.cuda.is_available()):
        raise RuntimeError('No CUDA device. Run within GPU Slurm allocation or --cpu for small smoke.')
    torch.set_num_threads(min(16, max(1, int(os.environ.get('SLURM_CPUS_PER_TASK', '8')))))
    print('device:', device, 'variant:', args.variant, flush=True)
    checkpoint = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
    if 'model' not in checkpoint or 'args' not in checkpoint:
        raise RuntimeError('Expected full original Uni-Core checkpoint with model and args')
    from argparse import Namespace
    cp_args = torch.load(EVAL_REFERENCE_CHECKPOINT, map_location='cpu', weights_only=False)['args']
    if isinstance(cp_args, dict):
        cp_args = Namespace(**cp_args)
    if not hasattr(cp_args, 'task') or not hasattr(cp_args, 'loss'):
        raise RuntimeError('Checkpoint args lack task/loss. Inspect real checkpoint schema.')
    cp_args.data = str(args.data_dir.resolve())
    cp_args.mode = 'train'
    if hasattr(cp_args, 'num_workers'):
        cp_args.num_workers = 0
    if hasattr(cp_args, 'user_dir'):
        cp_args.user_dir = str(expected.resolve())
    task = tasks.setup_task(cp_args)
    import random
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(args.seed)
    model = task.build_model(cp_args)
    if args.variant == 'trained':
        load_info = model.load_state_dict(checkpoint['model'], strict=True)
        print('Checkpoint load (strict):', load_info, flush=True)
    criterion = task.build_loss(cp_args)
    model = model.to(device).eval()
    criterion = criterion.to(device).eval()
    del checkpoint
    task.load_dataset(args.split, epoch=1)
    dataset = task.dataset(args.split)
    if hasattr(dataset, 'set_epoch'):
        dataset.set_epoch(1)
    meta = pd.read_csv(args.manifest_dir / 'sample_metadata.csv', keep_default_na=False)
    info = json.loads((args.manifest_dir / 'sampling_summary.json').read_text())
    if len(meta) != info['n'] or len(dataset) != info['source_lmdb_records']:
        raise RuntimeError('Manifest length / dataset count mismatch!')
    if not np.array_equal(meta.sample_order.to_numpy(), np.arange(len(meta))):
        raise RuntimeError('Manifest order invalid')
    if not (meta.lmdb_index.astype(str) == meta.lmdb_key.astype(str)).all():
        raise RuntimeError('Non-numeric manifest LMDB mapping; recreate using NEW prepare_subset.py')
    if args.limit:
        meta = meta.iloc[:args.limit].copy()
    if meta.empty:
        raise RuntimeError('No records')
    if meta.lmdb_index.min() < 0 or meta.lmdb_index.max() >= len(dataset):
        raise RuntimeError('Invalid LMDB index in manifest')
    N = len(meta)
    dims = int(getattr(cp_args, 'encoder_embed_dim', 512))
    maps = {}
    if not args.skip_embeddings:
        for name in ['cls_view0', 'cls_view1', 'projected_view0', 'projected_view1']:
            maps[name] = np.lib.format.open_memmap(args.out / f'{name}.npy', mode='w+', dtype='float32', shape=(N, dims))
    captures = {}

    def enc_hook(mod, inputs, output):
        enc = output[0] if isinstance(output, (tuple, list)) else output
        captures['encoder_cls'] = enc[:, 0, :].detach()

    def model_hook(mod, inputs, output):
        if not isinstance(output, (tuple, list)) or len(output) != 6:
            raise RuntimeError('Unexpected HEART model return structure')
        captures['logits'] = output[0].detach() if output[0] is not None else None
        captures['distance'] = output[1].detach() if output[1] is not None else None
        captures['coord'] = output[2].detach() if output[2] is not None else None
        captures['projected'] = output[5].detach()
    h1 = model.encoder.register_forward_hook(enc_hook)
    h2 = model.register_forward_hook(model_hook)
    batch_fields = ['start', 'n_molecules', 'n_views', 'masked_token_cnt', 'masked_token_hit', 'masked_acc', 'masked_token_loss', 'masked_coord_loss', 'masked_dist_loss', 'contrastive_loss', 'official_weighted_loss', 'coord_pred_rmse', 'coord_noisy_rmse', 'coord_denoising_improvement', 'distance_pred_rmse_original_units', 'positive_projection_cosine', 'negative_projection_cosine', 'contrastive_top1_view0_to_view1', 'contrastive_top1_view1_to_view0', 'augmentation_token_mismatch_fraction']
    mol_fields = ['sample_order', 'lmdb_index', 'view', 'masked_atom_count', 'masked_atom_hits', 'masked_atom_accuracy', 'coord_pred_rmse', 'coord_noisy_rmse']
    collected = []
    input_hash = hashlib.sha256()
    def hash_tree(obj):
        if isinstance(obj, torch.Tensor):
            a = obj.detach().cpu().contiguous().numpy(); input_hash.update(str(a.shape).encode()); input_hash.update(str(a.dtype).encode()); input_hash.update(a.tobytes())
        elif isinstance(obj, dict):
            for k in sorted(obj): input_hash.update(str(k).encode()); hash_tree(obj[k])
        elif isinstance(obj, (list,tuple)):
            for v in obj: hash_tree(v)
        else: input_hash.update(repr(obj).encode())
    masked_hits = masked_counts = 0
    atom_target_counts = Counter()
    atom_correct_counts = Counter()
    geometry_sq = geometry_noisy_sq = geometry_nums = 0
    pos_vals = []
    neg_vals = []
    retrieval_hits = 0
    retrieval_possible = 0
    sampled_token_view_changes = []
    status = 'failed'
    try:
        with (args.out / 'objective_batches.csv').open('w', newline='') as bf, (args.out / 'molecule_diagnostics.csv').open('w', newline='') as mf:
            bw = csv.DictWriter(bf, fieldnames=batch_fields)
            bw.writeheader()
            mw = csv.DictWriter(mf, fieldnames=mol_fields)
            mw.writeheader()
            with torch.inference_mode():
                for begin in range(0, N, args.batch_size):
                    stop = min(begin + args.batch_size, N)
                    sub = meta.iloc[begin:stop]
                    ids = sub.lmdb_index.astype(int).tolist()
                    b = len(ids)
                    sample = dataset.collater([dataset[i] for i in ids])
                    if not sample:
                        raise RuntimeError('Empty batch at ' + str(begin))
                    if 'smi_name' in sample.get('target', {}):
                        got = sample['target']['smi_name']
                        expect = sub.smiles.astype(str).tolist()
                        if len(got) != 2 * b:
                            raise RuntimeError('Expected 2-batch SMILES target names')
                        for j in range(b):
                            x = got[j]
                            if isinstance(x, bytes):
                                x = x.decode('utf8', errors='replace')
                            if str(x).strip() != expect[j].strip():
                                raise RuntimeError(f'SMILES misalignment at id={ids[j]}: manifest={expect[j]!r}, task={x!r}')
                    hash_tree(sample)
                    sample = heart_eval_move(sample, device)
                    captures.clear()
                    loss, sample_size, log = criterion(model, sample)
                    if not all((k in captures for k in ('encoder_cls', 'projected', 'logits'))):
                        raise RuntimeError('Model/encoder hooks missing; check code version')
                    masked = sample['target']['tokens_target'].ne(criterion.padding_idx)
                    if masked.shape[0] != 2 * b:
                        raise RuntimeError(f'Expected two views {2 * b}, got {masked.shape[0]}')
                    target = sample['target']['tokens_target']
                    logits = captures['logits']
                    if logits is None or logits.shape[0] != int(masked.sum()):
                        raise RuntimeError('Masked logits do not align with masked positions')
                    pred = logits.argmax(dim=-1)
                    truth = target[masked]
                    total = int(masked.sum().item())
                    hit = int((pred == truth).sum().item())
                    masked_hits += hit
                    masked_counts += total
                    target_ids = truth.detach().cpu().tolist()
                    correct_ids = truth[pred == truth].detach().cpu().tolist()
                    atom_target_counts.update((int(x) for x in target_ids))
                    atom_correct_counts.update((int(x) for x in correct_ids))
                    pred_filled = torch.full_like(target, -999)
                    pred_filled[masked] = pred
                    each_cnt = masked.sum(dim=1)
                    each_hit = ((pred_filled == target) & masked).sum(dim=1)
                    coord_pred = captures.get('coord')
                    coord_target = sample['target']['coord_target']
                    src_coord = sample['net_input']['src_coord']
                    if coord_pred is not None:
                        err = (coord_pred.float() - coord_target.float()).pow(2).sum(dim=-1)
                        noisy = (src_coord.float() - coord_target.float()).pow(2).sum(dim=-1)
                        per_coord = (err * masked).sum(dim=1) / (each_cnt.clamp_min(1) * 3)
                        per_noisy = (noisy * masked).sum(dim=1) / (each_cnt.clamp_min(1) * 3)
                        per_coord_rmse = per_coord.sqrt().cpu().numpy()
                        per_noisy_rmse = per_noisy.sqrt().cpu().numpy()
                        geometry_sq += float((err * masked).sum().item())
                        geometry_noisy_sq += float((noisy * masked).sum().item())
                        geometry_nums += 3 * total
                        coord_rmse = float((err[masked].mean() / 3).sqrt().item()) if total else float('nan')
                        noisy_rmse = float((noisy[masked].mean() / 3).sqrt().item()) if total else float('nan')
                    else:
                        coord_rmse = noisy_rmse = float('nan')
                        per_coord_rmse = per_noisy_rmse = np.full(2 * b, np.nan)
                    dist_pred = captures.get('distance')
                    d_rmse = float('nan')
                    if dist_pred is not None:
                        pairmask = masked.unsqueeze(-1) & sample['net_input']['src_tokens'].ne(criterion.padding_idx).unsqueeze(1)
                        dp = dist_pred.float() * criterion.dist_std + criterion.dist_mean
                        dt = sample['target']['distance_target'].float()
                        if pairmask.any():
                            d_rmse = float((dp[pairmask] - dt[pairmask]).pow(2).mean().sqrt().item())
                    z = captures['projected']
                    cls = captures['encoder_cls']
                    if z.shape != (2 * b, dims) or cls.shape != (2 * b, dims):
                        raise RuntimeError(f'Embedding shape mismatch: cls={cls.shape}, proj={z.shape}; expected 2*b x {dims}')
                    z0 = z[:b].float()
                    z1 = z[b:].float()
                    similarities = z0 @ z1.T
                    same = similarities.diag()
                    pos_vals.extend(same.detach().cpu().tolist())
                    negative = similarities[~torch.eye(b, dtype=torch.bool, device=device)]
                    neg = negative.mean().item() if negative.numel() else float('nan')
                    if negative.numel():
                        neg_vals.append(float(neg))
                    top01 = float((similarities.argmax(dim=1) == torch.arange(b, device=device)).float().mean().item())
                    top10 = float((similarities.argmax(dim=0) == torch.arange(b, device=device)).float().mean().item())
                    retrieval_hits += int(round(top01 * b)) + int(round(top10 * b))
                    retrieval_possible += 2 * b
                    tok = sample['net_input']['src_tokens']
                    token_mismatch = float((tok[:b] != tok[b:]).float().mean().item())
                    sampled_token_view_changes.append(token_mismatch)
                    if maps:
                        for name, arr in [('cls_view0', cls[:b]), ('cls_view1', cls[b:]), ('projected_view0', z0), ('projected_view1', z1)]:
                            maps[name][begin:stop] = arr.float().cpu().numpy()
                    for v in range(2):
                        for j in range(b):
                            k = v * b + j
                            nmask = int(each_cnt[k].item())
                            ncorrect = int(each_hit[k].item())
                            mw.writerow({'sample_order': int(sub.iloc[j].sample_order), 'lmdb_index': int(ids[j]), 'view': v, 'masked_atom_count': nmask, 'masked_atom_hits': ncorrect, 'masked_atom_accuracy': ncorrect / nmask if nmask else '', 'coord_pred_rmse': float(per_coord_rmse[k]), 'coord_noisy_rmse': float(per_noisy_rmse[k])})

                    def logged(key):
                        return heart_eval_float_scalar(log.get(key))
                    row = {'start': begin, 'n_molecules': b, 'n_views': 2 * b, 'masked_token_cnt': total, 'masked_token_hit': hit, 'masked_acc': hit / total if total else None, 'masked_token_loss': logged('masked_token_loss'), 'masked_coord_loss': logged('masked_coord_loss'), 'masked_dist_loss': logged('masked_dist_loss'), 'contrastive_loss': logged('contrastive_loss'), 'official_weighted_loss': logged('loss'), 'coord_pred_rmse': coord_rmse, 'coord_noisy_rmse': noisy_rmse, 'coord_denoising_improvement': noisy_rmse - coord_rmse, 'distance_pred_rmse_original_units': d_rmse, 'positive_projection_cosine': float(same.mean().item()), 'negative_projection_cosine': float(neg), 'contrastive_top1_view0_to_view1': top01, 'contrastive_top1_view1_to_view0': top10, 'augmentation_token_mismatch_fraction': token_mismatch}
                    bw.writerow(row)
                    bf.flush()
                    mf.flush()
                    collected.append(row)
                    if stop % max(128, args.batch_size) < args.batch_size or stop == N:
                        print(f'[{args.variant}] {stop:,}/{N:,} masked_acc={masked_hits / masked_counts:.4%} coordRMSE={coord_rmse:.4f} vs noisy={noisy_rmse:.4f}', flush=True)
                    del sample, captures['encoder_cls'], captures['projected'], captures['logits']
            for m in maps.values():
                m.flush()
        status = 'complete'
    finally:
        h1.remove()
        h2.remove()
    avgs = {key: heart_eval_avg([r[key] for r in collected if r.get(key) is not None and np.isfinite(r[key])]) for key in ['masked_token_loss', 'masked_coord_loss', 'masked_dist_loss', 'contrastive_loss', 'official_weighted_loss', 'positive_projection_cosine', 'negative_projection_cosine', 'contrastive_top1_view0_to_view1', 'contrastive_top1_view1_to_view0', 'distance_pred_rmse_original_units']}
    atom_csv = args.out / 'atom_token_class_accuracy.csv'
    symbol_names = getattr(task.dictionary, 'symbols', None) if hasattr(task, 'dictionary') else None
    with atom_csv.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['token_id', 'token_symbol', 'masked_targets', 'correct', 'accuracy', 'target_fraction'])
        writer.writeheader()
        for token_id, n in atom_target_counts.most_common():
            symbol = str(symbol_names[token_id]) if symbol_names is not None and token_id < len(symbol_names) else str(token_id)
            correct = atom_correct_counts[token_id]
            writer.writerow({'token_id': token_id, 'token_symbol': symbol, 'masked_targets': n, 'correct': correct, 'accuracy': correct / n if n else None, 'target_fraction': n / masked_counts if masked_counts else None})
    summary = {'status': status, 'evaluation_input_sha256': input_hash.hexdigest(), 'utc': heart_eval_now(), 'variant': args.variant, 'n_molecules': N, 'n_views': 2 * N, 'batches': len(collected), 'masked_tokens': masked_counts, 'masked_token_hits': masked_hits, 'masked_atom_accuracy': masked_hits / masked_counts if masked_counts else None, 'masked_atom_majority_class_baseline_accuracy': max(atom_target_counts.values()) / masked_counts if masked_counts else None, 'masked_atom_majority_class_token_id': atom_target_counts.most_common(1)[0][0] if atom_target_counts else None, 'atom_class_csv': str(atom_csv), 'coord_pred_rmse': float(np.sqrt(geometry_sq / geometry_nums)) if geometry_nums else None, 'coord_noisy_rmse': float(np.sqrt(geometry_noisy_sq / geometry_nums)) if geometry_nums else None, 'coord_rmse_reduction_fraction': float(1 - np.sqrt(geometry_sq / geometry_noisy_sq)) if geometry_noisy_sq else None, 'contrastive_top1_both_directions': retrieval_hits / retrieval_possible if retrieval_possible else None, 'positive_projection_cosine_mean': heart_eval_avg(pos_vals), 'negative_projection_cosine_mean_of_batches': heart_eval_avg(neg_vals), 'criterion_mean_over_batches': avgs, 'batch_size_molecules': args.batch_size, 'contrastive_note': 'NT-Xent is batch-size-dependent; top1 only within a batch of this size, not global 10k retrieval. The 0.624 historic validation loss at batch 448 must not be directly compared numerically against a different-batch-size 10k loss.', 'geometry_note': 'RMSE is in the coordinate units of transformed LMDB target (centering normalized, not necessarily global alignment). SmoothL1 is the training objective.', 'source_lmdb': str((args.data_dir / f'{args.split}.lmdb').resolve()), 'manifest': str((args.manifest_dir / 'sample_metadata.csv').resolve()), 'manifest_sha256': heart_eval_sha256_file(args.manifest_dir / 'sample_metadata.csv'), 'checkpoint': str(args.checkpoint.resolve()), 'checkpoint_sha256': heart_eval_sha256_file(args.checkpoint), 'checkpoint_args': {name: getattr(cp_args, name, None) for name in ['task', 'loss', 'arch', 'encoder_embed_dim', 'encoder_layers', 'encoder_attention_heads', 'contrastive_view_mode', 'mask_prob', 'noise', 'noise_type', 'masked_token_loss', 'masked_coord_loss', 'masked_dist_loss', 'contrastive_loss', 'only_polar', 'max_atoms']}, 'model_git': heart_eval_git_info(args.repo), 'embedding_files': list(maps.keys()), 'raw_metric_definitions': {'masked_acc': 'global sum of correct predictions divided by all masked token targets (both augmented views)', 'coord_rmse': 'RMSE between predicted coords and target 3D coords at masked atoms; compare with noisy input RMSE', 'contrastive_top1': 'in-batch diagonal pairing rank-1 for projected normalized representations', 'original_loss': 'batch-mean of original loss outputs; contrastive scale depends on batch size'}, 'warning': 'Pretraining-objective evaluation is not evidence of downstream lipid RNA-delivery accuracy. Data leakage must be audited separately.'}
    (args.out / 'evaluation_summary.json').write_text(json.dumps(summary, indent=2, default=str) + '\n')
    print('COMPLETE:', args.out / 'evaluation_summary.json', flush=True)

In [ ]:
RESULT_ROOT=OUT/'frozen_evaluation'/f'n{N_EVAL}_batch{BATCH}_seed{EVAL_SEED}'
if RUN_FROZEN_EVAL:
    import torch, gc
    assert torch.cuda.is_available(), 'Use allocated GPU Jupyter kernel'
    assert TRAIN_SETTINGS_REVIEWED and VERIFIED_UPDATES.get('A')==VERIFIED_UPDATES.get('B')==5000, 'Confirm matched settings and actual checkpoint updates before evaluation'
    assert set(CHECKPOINTS)=={'General','A','B'}
    assert len(pd.read_csv(GENERAL_MANIFEST/'sample_metadata.csv'))>=N_EVAL
    for domain,(data,manifest,n) in PANELS.items():
        for m,ck in CHECKPOINTS.items():
            dest=RESULT_ROOT/domain/m
            if (dest/'evaluation_summary.json').is_file():
                old=read_json(dest/'evaluation_summary.json')
                if old.get('status')!='complete' or old.get('checkpoint_sha256')!=sha(ck) or old.get('manifest_sha256')!=sha(manifest/'sample_metadata.csv') or old.get('n_molecules')!=n or old.get('batch_size_molecules')!=BATCH or not old.get('evaluation_input_sha256'):raise RuntimeError('Cached result incompatible: '+str(dest))
                print('Reuse',dest);continue
            heart_eval_main(Namespace(repo=REPO,data_dir=data,checkpoint=ck,manifest_dir=manifest,out=dest,unicore_dir=UNICORE,limit=n,batch_size=BATCH,seed=EVAL_SEED,variant='trained',cpu=False,skip_embeddings=False,split='valid'))
            gc.collect();torch.cuda.empty_cache()
else:print('GPU evaluation disabled.')

## 5 | Matched domain comparison and paired uncertainty
Only complete triplets with identical manifest and actual input fingerprints are compared. Compare A vs B within each domain; compare General vs A/B for retention. Masked token accuracy, coordinate RMSE, distance RMSE, NT-Xent, positive/negative similarity and top-1. Batch-level paired bootstrap is descriptive conditional on a fixed pair of trained checkpoints (not training-seed uncertainty). The same batch is resampled jointly; aggregate coordinate RMSE is weighted by masked coordinate count. Projection cosine and top-1 are independently computed from normalized stored vectors. PCA uses separate fits for each model and the same descriptor color range; independently fitted axes are not directly comparable, so visual displacement is not proof of chemical improvement.

In [ ]:
from sklearn.decomposition import PCA
rows=[]; paired=[]
for domain in ['General','Ugi','LNPDB']:
    dirs={m:RESULT_ROOT/domain/m for m in ['General','A','B']}
    if not all((d/'evaluation_summary.json').is_file() for d in dirs.values()):print('Awaiting full triplet:',domain);continue
    ss={m:read_json(d/'evaluation_summary.json') for m,d in dirs.items()}
    for m in dirs:
        if m not in CHECKPOINTS or not CHECKPOINTS[m].is_file() or ss[m].get('checkpoint_sha256')!=sha(CHECKPOINTS[m]):raise ValueError(f'{domain}: checkpoint provenance mismatch for {m}')
        if ss[m].get('manifest_sha256')!=sha(ss[m]['manifest']):raise ValueError(f'{domain}: manifest changed for {m}')
    for field in ['manifest_sha256','evaluation_input_sha256','n_molecules','batch_size_molecules']:
        values=[s.get(field) for s in ss.values()]
        if None in values or len(set(values))!=1:raise ValueError(f'{domain}: unmatched {field}: {values}')
    if any(s['status']!='complete' for s in ss.values()):raise ValueError('Incomplete evaluation')
    tables={m:pd.read_csv(d/'molecule_diagnostics.csv').sort_values(['sample_order','view']).reset_index(drop=True) for m,d in dirs.items()}
    batches={m:pd.read_csv(d/'objective_batches.csv').sort_values('start').reset_index(drop=True) for m,d in dirs.items()}
    for m in ['A','B']:
        for cols,ts in [(['sample_order','lmdb_index','view','masked_atom_count','coord_noisy_rmse'],tables),(['start','n_molecules','masked_token_cnt'],batches)]:
            if not ts[m][cols].equals(ts['General'][cols]):raise ValueError(f'{domain}: paired records disagree {m} {cols}')
    for m,s in ss.items():
        d=dirs[m];b=batches[m];z0=np.load(d/'projected_view0.npy');z1=np.load(d/'projected_view1.npy')
        n0=np.linalg.norm(z0,axis=1);n1=np.linalg.norm(z1,axis=1)
        if not np.isfinite(z0).all() or not np.isfinite(z1).all() or np.any(n0==0) or np.any(n1==0):raise ValueError('Invalid projections')
        z0=z0/n0[:,None];z1=z1/n1[:,None]
        pos=[];neg=[];top=[]
        for _,r in b.iterrows():
            i=int(r.start);j=i+int(r.n_molecules);sim=z0[i:j]@z1[i:j].T; k=len(sim);pos.extend(np.diag(sim));neg.extend(sim[~np.eye(k,dtype=bool)]);top.extend(sim.argmax(0)==np.arange(k));top.extend(sim.argmax(1)==np.arange(k))
        rows.append({'domain':domain,'model':m,'n':s['n_molecules'],'atom_accuracy':s['masked_atom_accuracy'],'coord_rmse':s['coord_pred_rmse'],'noisy_rmse':s['coord_noisy_rmse'],'distance_rmse_batch_mean':s['criterion_mean_over_batches']['distance_pred_rmse_original_units'],'nt_xent':s['criterion_mean_over_batches']['contrastive_loss'],'positive_cosine':np.mean(pos),'negative_cosine':np.mean(neg),'top1':np.mean(top),'projection_norm_mean':np.mean(np.r_[n0,n1]),**{k:s['criterion_mean_over_batches'].get(k,np.nan) for k in ['official_weighted_loss','masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss']}})
    # Paired batch bootstrap, conditional on these trained checkpoints and inputs.
    rng=np.random.default_rng(71); idx=rng.integers(len(batches['A']),size=(2000,len(batches['A'])))
    def metrics(t,ix):
        w=t.masked_token_cnt.to_numpy()[ix]; hit=t.masked_token_hit.to_numpy()[ix]; r=t.coord_pred_rmse.to_numpy()[ix]
        result={'atom_accuracy':hit.sum(axis=-1)/w.sum(axis=-1),'coord_rmse':np.sqrt((r**2*w).sum(axis=-1)/w.sum(axis=-1))}
        for k in ['official_weighted_loss','masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','distance_pred_rmse_original_units']:
            if k in t and np.isfinite(t[k]).all():result[k]=t[k].to_numpy()[ix].mean(axis=-1)
        return result
    for left,right in [('General','A'),('General','B'),('A','B')]:
        a,b=batches[left],batches[right]
        if len(a)<2 or (a.masked_token_cnt<=0).any() or (b.masked_token_cnt<=0).any():raise ValueError('Invalid paired batches')
        ma,mb=metrics(a,idx),metrics(b,idx);pa,pb=metrics(a,np.arange(len(a))),metrics(b,np.arange(len(b)))
        for key in ma.keys() & mb.keys():
            d=mb[key]-ma[key];point=float(pb[key]-pa[key])
            if not np.isfinite(d).all():continue
            paired.append({'domain':domain,'metric':key,'comparison':f'{right} - {left}','delta':point,'B_minus_A':point if (left,right)==('A','B') else np.nan,'conditional_ci_low':np.quantile(d,.025),'conditional_ci_high':np.quantile(d,.975)})
    # Independently fitted PCA per model, preserve actual explained variance; no percentile clipping.
    # Use independent fits rather than inferring cross-model point movement from latent rotations.
    from matplotlib.colors import LinearSegmentedColormap,Normalize
    cmap=LinearSegmentedColormap.from_list('warm',['#F4E4A9','#EDC65F','#E6A34C','#DC8051','#CF5B50','#B54A55','#7C546D'])
    from rdkit import Chem
    from rdkit.Chem import Descriptors
    meta=pd.read_csv(ss['General']['manifest']).iloc[:ss['General']['n_molecules']]
    ms=[Chem.MolFromSmiles(x) for x in meta.smiles];props={'Heavy atoms':np.array([x.GetNumHeavyAtoms() if x else np.nan for x in ms]),'cLogP':np.array([Descriptors.MolLogP(x) if x else np.nan for x in ms])}
    fig,axs=plt.subplots(2,3,figsize=(16,9),layout='constrained')
    for j,(m,d) in enumerate(dirs.items()):
        X=np.load(d/'cls_view0.npy');pca=PCA(n_components=2,svd_solver='randomized',random_state=17);xy=pca.fit_transform(X)
        for i,(prop,z) in enumerate(props.items()):
            ax=axs[i,j];ok=np.isfinite(z);assert ok.any();sc=ax.scatter(*xy[ok].T,c=z[ok],s=5,alpha=.8,cmap=cmap,norm=Normalize(z[ok].min(),z[ok].max()),linewidths=0);fig.colorbar(sc,ax=ax,label=prop);ax.set(title=f'{m} — {prop}',xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})');ax.text(-.10,1.05,'abcdef'[i*3+j],transform=ax.transAxes,fontweight='bold')
    fig.suptitle(f'{domain}: CLS representation comparison',fontweight='bold');save(fig,f'CMRI_Model_AB_{domain}_PCA')
summary=pd.DataFrame(rows);delta=pd.DataFrame(paired)
summary.to_csv(OUT/'domain_metrics.csv',index=False);delta.to_csv(OUT/'paired_batch_deltas.csv',index=False)
if len(summary):
    display(summary.round(5));display(delta.round(5))
    keys=[('atom_accuracy','Masked atom accuracy ↑'),('coord_rmse','Coordinate RMSE (Å) ↓'),('nt_xent','NT-Xent ↓'),('top1','In-batch top-1 ↑'),('positive_cosine','Positive cosine ↑'),('negative_cosine','Negative cosine')]
    fig,axs=plt.subplots(2,3,figsize=(16,8),layout='constrained')
    for k,(ax,(metric,title)) in enumerate(zip(axs.flat,keys)):
        q=summary.pivot(index='domain',columns='model',values=metric).reindex(columns=['General','A','B']);q.plot.bar(ax=ax,color=[COLORS[m] for m in q.columns],rot=0);ax.set(title=title,xlabel='');ax.text(-.10,1.05,'abcdef'[k],transform=ax.transAxes,fontweight='bold');ax.legend(frameon=False,fontsize=12)
    fig.suptitle('CMRI_Model matched held-out evaluation',fontweight='bold');save(fig,'CMRI_Model_AB_domain_comparison')
else:print('No complete matched domain results yet.')
if len(summary):
    loss_keys=['official_weighted_loss','masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','distance_rmse_batch_mean']
    display(summary[['domain','model']+loss_keys])
    summary[['domain','model']+loss_keys].to_csv(OUT/'objective_loss_comparison.csv',index=False)
    fig,axs=plt.subplots(2,3,figsize=(16,8),layout='constrained')
    for ax,k in zip(axs.flat,loss_keys):
        q=summary.pivot(index='domain',columns='model',values=k).reindex(columns=['General','A','B']);q.plot.bar(ax=ax,color=[COLORS[m] for m in q.columns],rot=0);ax.set(title=k.replace('_',' '),xlabel='');ax.legend(frameon=False)
    fig.suptitle('Matched objective losses and distance RMSE');save(fig,'CMRI_Model_AB_objective_losses')


## 6 | Optional downstream transfer (original chemistry benchmark code)
Reuse the same previously prepared independent 3D input, same original chemistry_tests.py and seeds 17,31,53, fractions 0.2 and 1.0. Existing general benchmark code calls frozen inference and trains only Ridge probes. Run on allocated GPU. A/B use the exact same input and code; if the checkpoint's mixed-task args are incompatible, stop (do not edit checkpoints). Verify generated split assignments or split hashes if the helper exposes them; matching seeds alone do not prove identical splits if the input rows differ. Audit source overlap across all training stages separately.

In [ ]:
CHEM_ROOT=AUDIT/'chemistry_slide_evidence'
EXTERNAL_LMDB=CHEM_ROOT/'benchmark_generated_lmdb'
CHEM_CODE=next((d for d in [Path.cwd(),Path.cwd()/'HEART_Chemistry_Slide_Evidence',HOME/'HEART_Chemistry_Slide_Evidence'] if (d/'chemistry_tests.py').is_file()),None)
DOWN_OUT=OUT/'downstream'
if RUN_DOWNSTREAM:
    import torch
    assert TRAIN_SETTINGS_REVIEWED and VERIFIED_UPDATES.get('A')==VERIFIED_UPDATES.get('B')==5000
    assert set(CHECKPOINTS)=={'General','A','B'}
    assert torch.cuda.is_available() and CHEM_CODE is not None
    assert all((EXTERNAL_LMDB/x).is_file() for x in ['valid.lmdb','dict.txt','external_manifest.csv'])
    sys.path.insert(0,str(CHEM_CODE));import chemistry_tests as chemistry
    for m,ck in CHECKPOINTS.items():
        em=DOWN_OUT/m/'external_embeddings';dest=DOWN_OUT/m/'benchmarks';receipt=DOWN_OUT/m/'receipt.json'
        spec={'checkpoint_sha256':sha(ck),'manifest_sha256':sha(EXTERNAL_LMDB/'external_manifest.csv'),'valid_lmdb_sha256':sha(EXTERNAL_LMDB/'valid.lmdb'),'chemistry_code_sha256':sha(CHEM_CODE/'chemistry_tests.py'),'seeds':[17,31,53],'fractions':[.2,1.]}
        if receipt.exists():
            if read_json(receipt)!=spec or not (dest/'benchmark_all_runs.csv').exists():raise RuntimeError('Incompatible cached benchmark')
            continue
        if em.exists() or dest.exists():raise RuntimeError('Partial downstream output; inspect before rerun: '+str(em.parent))
        em.mkdir(parents=True);dest.mkdir(parents=True)
        chemistry.generate_external_embeddings(REPO,UNICORE,ck,EXTERNAL_LMDB,em,batch_size=8)
        chemistry.external_property_evidence(EXTERNAL_LMDB,em,dest,audit_dir=AUDIT/'full_ligand_audit/raw',seeds=(17,31,53),fractions=(.2,1.0))
        assert (dest/'benchmark_all_runs.csv').is_file()
        receipt.write_text(json.dumps(spec,indent=2))
else:print('Downstream inference/probes disabled.')
frames=[];specs=[]
for m in ['General','A','B']:
    d=DOWN_OUT/m; p=d/'benchmarks/benchmark_all_runs.csv'
    if p.is_file() and (d/'receipt.json').is_file():
        rec=read_json(d/'receipt.json');specs.append({k:v for k,v in rec.items() if k!='checkpoint_sha256'});frames.append(pd.read_csv(p).assign(checkpoint_model=m))
if len(frames)==3:
    assert specs[0]==specs[1]==specs[2]
    q=pd.concat(frames,ignore_index=True);q=q[np.isclose(q.train_fraction,1.0)]
    q.to_csv(OUT/'downstream_all_runs.csv',index=False)
    if 'model' in q and 'benchmark' in q and 'rmse' in q and 'seed' in q:
        print('Inspect actual method names and split provenance before declaring paired results:');display(q.groupby(['checkpoint_model','benchmark','model']).rmse.agg(['mean','std','count']))
        print('Seeds:');display(q.groupby(['checkpoint_model','benchmark','model']).seed.agg(list))
        # Select the pretrained CLS method only; fail closed on ambiguous naming.
        method=q.model.astype(str).str.lower()
        frozen=q[method.str.contains('cls|frozen|pretrain|heart',regex=True)&~method.str.contains('random|untrained',regex=True)].copy()
        if len(frozen) and set(frozen.checkpoint_model)=={'General','A','B'} and not frozen.duplicated(['checkpoint_model','benchmark','seed']).any():
            groups=frozen.groupby(['checkpoint_model','benchmark']).seed.agg(lambda x:tuple(sorted(x)))
            if len(groups)==9 and all(tuple(map(int,v))==(17,31,53) for v in groups):
                stat=frozen.groupby(['benchmark','checkpoint_model']).rmse.agg(['mean','std','count']);display(stat);stat.to_csv(OUT/'downstream_frozen_cls_summary.csv')
                fig,axs=plt.subplots(1,3,figsize=(14,4.5),layout='constrained')
                for i,(ax,(name,g)) in enumerate(zip(axs,stat.groupby(level=0))):
                    g=g.droplevel(0).reindex(['General','A','B']);ax.bar(g.index,g['mean'],yerr=g['std'],capsize=3,color=[COLORS[m] for m in g.index]);ax.set(title=str(name),ylabel='Scaffold-held-out RMSE');ax.text(-.10,1.05,'abc'[i],transform=ax.transAxes,fontweight='bold')
                fig.suptitle('Frozen CLS downstream transfer',fontweight='bold');save(fig,'CMRI_Model_AB_downstream')
            else:print('Incomplete or unexpected seed grid; no summary plot.')
        else:print('Ambiguous frozen CLS method or duplicate seed records; inspect method names before summary plotting.')
else:print('Awaiting all three downstream outputs.')

## 7 | Fixed-panel checkpoint progression
Step 0 is the common General checkpoint. Missing intermediate checkpoints are skipped; curves are not interpolated measurements.

In [ ]:
# Evaluate saved intermediate checkpoints only; no training. Reuse final and baseline cache.
if RUN_CHECKPOINT_CURVES:
    import torch,gc
    progression=[]
    for domain,(data,manifest,n_samples) in PANELS.items():
        base=read_json(RESULT_ROOT/domain/'General'/'evaluation_summary.json')
        for model in ['A','B']:
            candidates={0:CHECKPOINTS['General'],5000:CHECKPOINTS[model]}
            for step in [1000,2000,3000,4000]:
                found=sorted(RUNS[model].rglob(f'checkpoint*_{step}.pt'))
                if len(found)==1:candidates[step]=found[0]
                else:print(model,step,'missing or ambiguous; skipped',found)
            for step,ck in sorted(candidates.items()):
                dest=RESULT_ROOT/domain/('General' if step==0 else model) if step in [0,5000] else RESULT_ROOT/'progression'/domain/model/str(step)
                if step not in [0,5000]:
                    cp=torch.load(ck,map_location='cpu',weights_only=False);hist=cp.get('optimizer_history') or [];actual=hist[-1].get('num_updates') if hist else cp.get('num_updates');del cp
                    if actual!=step:print('Update mismatch',ck,actual);continue
                if not (dest/'evaluation_summary.json').exists():
                    heart_eval_main(Namespace(repo=REPO,data_dir=data,checkpoint=ck,manifest_dir=manifest,out=dest,unicore_dir=UNICORE,limit=n_samples,batch_size=BATCH,seed=EVAL_SEED,variant='trained',cpu=False,skip_embeddings=False,split='valid'))
                    gc.collect();torch.cuda.empty_cache()
                s=read_json(dest/'evaluation_summary.json')
                if s.get('status')!='complete' or s.get('checkpoint_sha256')!=sha(ck) or any(s.get(k)!=base.get(k) for k in ['manifest_sha256','evaluation_input_sha256','n_molecules','batch_size_molecules']):raise ValueError('Unmatched progression cache: '+str(dest))
                progression.append({'domain':domain,'model':model,'step':step,'coord_rmse':s['coord_pred_rmse'],'atom_accuracy':s['masked_atom_accuracy'],**s['criterion_mean_over_batches']})
    progress=pd.DataFrame(progression);progress.to_csv(OUT/'checkpoint_progression.csv',index=False)
    for domain,q in progress.groupby('domain'):
        keys=['official_weighted_loss','masked_token_loss','masked_coord_loss','masked_dist_loss','contrastive_loss','coord_rmse','distance_pred_rmse_original_units','atom_accuracy']
        fig,axs=plt.subplots(2,4,figsize=(20,8),layout='constrained')
        for ax,k in zip(axs.flat,keys):
            for model,g in q.groupby('model'):
                g=g.sort_values('step');ax.plot(g.step,g[k],marker='o',label=model,color=COLORS[model])
            ax.set(title=k.replace('_',' '),xlabel='Continual updates');ax.legend(frameon=False)
        fig.suptitle(domain+' fixed-panel checkpoint progression');save(fig,'CMRI_Model_AB_'+domain+'_progression')


## 8 | Interpretation / decision rules
- B improves verified LNPDB held-out evaluation vs A: evidence of transfer to added chemical domain (not independent if holdout contamination remains).
- B retains Ugi and General while improving LNPDB: promising augmentation at matched budget; establish acceptable retention margins before claiming no degradation.
- B improves independent downstream properties: evidence of transferable CLS features, not LNP potency or a proof of a noise mechanism. Similar or worse performance can indicate limited exposure, distribution mismatch, or optimization effects; no conclusion from one seed alone.
- Lower mixed training loss alone is insufficient. NT-Xent is batch/temperature dependent; compare only identical fixed inputs and common criterion. Three original scaffold seeds measure probe split variability, not variability across continual-training seeds.
- A mechanism test needs a matched control (e.g. replace 5% Ugi with a size/property-matched non-LNPDB auxiliary chemical set; match exposure and seed) and ideally multiple training seeds. If PI means adding coordinate noise to LNPDB, that is a different augmentation requiring a controlled new training arm.